# Base-paper training and evaluation helper

This notebook clones the authors' official repository directly into Kaggle and executes its Python entry points. It does not clone this helper repository, duplicate the implementation, modify the authors' source files, or push/pull GitHub changes.

Select one stage per Kaggle run. Begin with `deberta_smoke`; do not start full training until its output has been reviewed.

In [1]:
# --------------------------- Stage control ---------------------------
from pathlib import Path
import csv, hashlib, importlib.metadata, json, os, platform, re, shlex, subprocess, sys
from datetime import datetime, timezone

ACTIVE_STAGE = "deberta_smoke"
VALID_STAGES = {
    "audit", "deberta_smoke", "deberta_full",
    "deberta_eval_squad", "deberta_eval_trivia",
    "generative_train", "generative_eval_squad", "generative_eval_trivia",
    "extraction_train",
    "removal_segmentation_squad", "removal_segmentation_trivia",
    "removal_extraction_squad", "removal_extraction_trivia",
    "final_asr_segmentation_squad", "final_asr_segmentation_trivia",
    "final_asr_extraction_squad", "final_asr_extraction_trivia",
}
if ACTIVE_STAGE not in VALID_STAGES: raise ValueError(f"Unknown stage: {ACTIVE_STAGE}")

OFFICIAL_REPO_URL = "https://github.com/LukeChen-go/indirect-pia-detection.git"
OFFICIAL_DIR = Path("/kaggle/working/official-indirect-pia-detection")
RESULTS_ROOT = Path("/kaggle/working/base-paper-results")
MODELS_ROOT = Path("/kaggle/working/base-paper-models")
SMOKE_DATA_ROOT = Path("/kaggle/working/base-paper-smoke-data")

DEBERTA_MODEL = "microsoft/deberta-v3-base"
GENERATIVE_BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
EXTRACTION_BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
TARGET_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

# Replace these with attached Kaggle output paths in later sessions when needed.
# Keep 'prompt' in the DeBERTa checkpoint path: the upstream evaluator uses that substring to select its discriminative adapter.
DEBERTA_CHECKPOINT = MODELS_ROOT / "prompt-discriminative-cls-deberta"
GENERATIVE_CHECKPOINT = MODELS_ROOT / "generative-class-llama32-1b"
EXTRACTION_CHECKPOINT = MODELS_ROOT / "ext-llama32-3b"

ATTACKS = ["none", "naive", "ignore", "escape_separation", "completion_real", "completion_realcmb"]
INJECTED_ATTACKS, POSITIONS, SEED = ATTACKS[1:], ["start", "middle", "end"], 42
print("Selected stage:", ACTIVE_STAGE)

Selected stage: deberta_smoke


## Clone the official implementation

All later commands use this clone as their working directory, so imports and datasets come directly from the authors' repository.

In [2]:
if OFFICIAL_DIR.exists():
    if not (OFFICIAL_DIR / ".git").is_dir(): raise RuntimeError(f"Not a Git clone: {OFFICIAL_DIR}")
    print("Using existing official clone:", OFFICIAL_DIR)
else:
    subprocess.run(["git", "clone", "--depth", "1", OFFICIAL_REPO_URL, str(OFFICIAL_DIR)], check=True)
OFFICIAL_COMMIT = subprocess.check_output(["git", "-C", str(OFFICIAL_DIR), "rev-parse", "HEAD"], text=True).strip()
os.chdir(OFFICIAL_DIR)
print("Official commit:", OFFICIAL_COMMIT)
print("Working directory:", Path.cwd())

Cloning into '/kaggle/working/official-indirect-pia-detection'...


Official commit: 1e9e0f00179adc154e9ca991b9cd3206c99f9588
Working directory: /kaggle/working/official-indirect-pia-detection


## Environment and minimal dependencies

The authors' complete requirements freeze contains machine-specific and CUDA-specific pins. This keeps Kaggle's working PyTorch/CUDA installation and pins only the user-space packages required by the official scripts.

In [3]:
MINIMAL_REQUIREMENTS = ["accelerate==1.2.1", "deepspeed==0.16.2", "openai==1.63.2",
                        "peft==0.14.0", "sentencepiece==0.2.0",
                        "tokenizers==0.21.0", "transformers==4.47.1"]

def package_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return "not installed"

def environment_snapshot():
    import torch
    devices = []
    for index in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(index)
        devices.append({"index": index, "name": p.name, "memory_gib": round(p.total_memory/1024**3, 2),
                        "compute_capability": f"{p.major}.{p.minor}"})
    probe = getattr(torch.cuda, "is_bf16_supported", None)
    return {"captured_at_utc": datetime.now(timezone.utc).isoformat(), "official_commit": OFFICIAL_COMMIT,
            "python": sys.version.replace("\n", " "), "platform": platform.platform(),
            "packages": {n: package_version(n) for n in ["torch","transformers","deepspeed","accelerate","peft","tokenizers"]},
            "cuda_available": torch.cuda.is_available(), "compiled_cuda_version": torch.version.cuda,
            "bf16_supported_probe": bool(probe()) if probe and torch.cuda.is_available() else False,
            "devices": devices}

SETUP_DIR = RESULTS_ROOT / "setup"; SETUP_DIR.mkdir(parents=True, exist_ok=True)
before = environment_snapshot()
(SETUP_DIR/"environment_before.json").write_text(json.dumps(before, indent=2)+"\n", encoding="utf-8")
print(json.dumps(before, indent=2))
if ACTIVE_STAGE != "audit":
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *MINIMAL_REQUIREMENTS]
    with (SETUP_DIR/"dependency_install.log").open("w", encoding="utf-8") as log:
        installed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    if installed.returncode:
        print("".join((SETUP_DIR/"dependency_install.log").read_text(errors="replace").splitlines(True)[-100:]))
        raise RuntimeError(f"Dependency installation failed: {installed.returncode}")
    after = environment_snapshot()
    (SETUP_DIR/"environment_after.json").write_text(json.dumps(after, indent=2)+"\n", encoding="utf-8")
    print(json.dumps(after, indent=2))
else: print("Audit selected: dependency installation skipped.")

{
  "captured_at_utc": "2026-10-07T16:44:00.048761+00:00",
  "official_commit": "1e9e0f00179adc154e9ca991b9cd3206c99f9588",
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.18.48+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "deepspeed": "not installed",
    "accelerate": "1.14.0",
    "peft": "0.20.0",
    "tokenizers": "0.23.1"
  },
  "cuda_available": true,
  "compiled_cuda_version": "12.8",
  "bf16_supported_probe": true,
  "devices": [
    {
      "index": 0,
      "name": "Tesla T4",
      "memory_gib": 14.56,
      "compute_capability": "7.5"
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "memory_gib": 14.56,
      "compute_capability": "7.5"
    }
  ]
}
{
  "captured_at_utc": "2026-10-07T16:45:37.548051+00:00",
  "official_commit": "1e9e0f00179adc154e9ca991b9cd3206c99f9588",
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.18.

## Gated-model authentication

Llama training and final target-model stages require a Kaggle secret named `HF_TOKEN`. The value is never printed.

In [4]:
GATED_STAGES = {"generative_train", "extraction_train",
    "final_asr_segmentation_squad", "final_asr_segmentation_trivia",
    "final_asr_extraction_squad", "final_asr_extraction_trivia"}
if ACTIVE_STAGE in GATED_STAGES:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if not token: raise RuntimeError("Create a Kaggle secret named HF_TOKEN.")
    os.environ["HF_TOKEN"] = token; del token
    print("HF_TOKEN loaded from Kaggle Secrets.")
else: print("No Hugging Face token is needed for this stage.")

No Hugging Face token is needed for this stage.


## Helper functions

These functions prepare the smoke subset, launch the official entry points, capture reproducibility evidence, and parse their text logs.

In [5]:
def utc_now(): return datetime.now(timezone.utc).isoformat()
def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()
def tail_text(path, count=120):
    p=Path(path)
    return "" if not p.exists() else "".join(p.read_text(encoding="utf-8",errors="replace").splitlines(True)[-count:])
def prepare_smoke_data():
    SMOKE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    specs=[(OFFICIAL_DIR/"data/crafted_instruction_data_alpaca.json",SMOKE_DATA_ROOT/"alpaca_64.json",64),
           (OFFICIAL_DIR/"data/crafted_instruction_data_context_squad.json",SMOKE_DATA_ROOT/"squad_16.json",16)]
    for src,dst,n in specs:
        rows=json.loads(src.read_text(encoding="utf-8")); dst.write_text(json.dumps(rows[:n],ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    return specs[0][1], specs[1][1]
def require_checkpoint(path,label):
    if not Path(path).exists(): raise FileNotFoundError(f"{label} checkpoint not found at {path}. Attach prior Kaggle output and update Stage control.")
def run_stage(stage, command, extra_env=None):
    out=RESULTS_ROOT/stage; out.mkdir(parents=True,exist_ok=True)
    for name in ["error.txt","console.log","upstream.log"]:
        p=out/name
        if p.exists(): p.unlink()
    command_text=shlex.join([str(x) for x in command]); (out/"command.txt").write_text(command_text+"\n",encoding="utf-8")
    config={"stage":stage,"official_repository":OFFICIAL_REPO_URL,"official_commit":OFFICIAL_COMMIT,"seed":SEED,
            "started_at_utc":utc_now(),"command":command_text,"scientific_result":stage!="deberta_smoke"}
    (out/"config.json").write_text(json.dumps(config,indent=2)+"\n",encoding="utf-8")
    (out/"environment.json").write_text(json.dumps(environment_snapshot(),indent=2)+"\n",encoding="utf-8")
    env=os.environ.copy(); env.update(extra_env or {})
    print(f"Running {stage}; full output -> {out/'console.log'}")
    with (out/"console.log").open("w",encoding="utf-8") as log:
        done=subprocess.run(command,cwd=OFFICIAL_DIR,env=env,stdout=log,stderr=subprocess.STDOUT)
    status={"stage":stage,"status":"success" if done.returncode==0 else "failed","return_code":done.returncode,"finished_at_utc":utc_now()}
    (out/"status.json").write_text(json.dumps(status,indent=2)+"\n",encoding="utf-8")
    if done.returncode: (out/"error.txt").write_text(f"Stage failed: {done.returncode}\n\n{tail_text(out/'console.log')}",encoding="utf-8")
    print(json.dumps(status,indent=2)); print(tail_text(out/"console.log")); return done.returncode,out
def ds_prefix(port): return [sys.executable,"-m","deepspeed.launcher.runner","--include","localhost:0","--master_port",str(port)]

START=re.compile(r"Attack Method (.*?), Side (.*?) Start"); START_ASR=re.compile(r"Attack Method (.*?), Defense Method (.*?),\s*Side (.*?), Begin")
PATTERNS={"score":re.compile(r"(?<!INJ )ACC:\s*([0-9.eE+-]+)"),"cost":re.compile(r"COST:\s*([0-9.eE+-]+)"),
          "asr":re.compile(r"ASR:\s*([0-9.eE+-]+)"),"removal_rate":re.compile(r"INJ ACC:\s*([0-9.eE+-]+)")}
def parse_log(mode,out,dataset,sample_count=900):
    rows=[]; current=None
    for line in (out/"upstream.log").read_text(encoding="utf-8",errors="replace").splitlines():
        start=(START_ASR if mode=="asr" else START).search(line)
        if start:
            current={"dataset":dataset,"attack":start.group(1).strip(),"sample_count":sample_count}
            current.update({"defense":start.group(2).strip(),"position":start.group(3).strip()} if mode=="asr" else {"position":start.group(2).strip()})
            rows.append(current); continue
        if current:
            for key,pattern in PATTERNS.items():
                m=pattern.search(line)
                if m: current[key]=float(m.group(1))
    for row in rows:
        if mode=="detection": row.update({"fpr":row.get("score") if row["attack"]=="none" else None,"tpr":None if row["attack"]=="none" else row.get("score")})
        elif mode=="removal": row["removal_rate"]=row.get("score")
        row.pop("score",None)
    if not rows: raise RuntimeError(f"No results parsed from {out/'upstream.log'}")
    (out/f"{mode}_results.json").write_text(json.dumps(rows,indent=2)+"\n",encoding="utf-8")
    keys=list(dict.fromkeys(k for row in rows for k in row))
    with (out/f"{mode}_results.csv").open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(rows)
    print(f"Parsed {len(rows)} rows"); return rows

## Run the selected official stage

Only the matching branch runs. Training uses GPU 0 to match the paper's single-GPU topology.

In [6]:
DATASETS={"squad":("data/crafted_instruction_data_squad_injection_qa.json","Inj-SQuAD"),
          "trivia":("data/crafted_instruction_data_tri_injection_qa.json","Inj-TriviaQA")}
last_stage_dir,last_return_code=None,0
if ACTIVE_STAGE=="audit":
    out=RESULTS_ROOT/"audit"; out.mkdir(parents=True,exist_ok=True); files=[]
    for p in sorted((OFFICIAL_DIR/"data").glob("*.json")):
        d=json.loads(p.read_text(encoding="utf-8")); files.append({"file":p.name,"records":len(d),"sha256":file_sha256(p),"sample_keys":sorted(d[0]) if d else []})
    report={"status":"ok","official_commit":OFFICIAL_COMMIT,"files":files}; (out/"data_summary.json").write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
    print(json.dumps(report,indent=2)); last_stage_dir=out
elif ACTIVE_STAGE in {"deberta_smoke","deberta_full"}:
    smoke=ACTIVE_STAGE=="deberta_smoke"
    ins,ctx=prepare_smoke_data() if smoke else (OFFICIAL_DIR/"data/crafted_instruction_data_alpaca.json",OFFICIAL_DIR/"data/crafted_instruction_data_context_squad.json")
    ckpt=MODELS_ROOT/"deberta-smoke" if smoke else DEBERTA_CHECKPOINT
    if ckpt.exists(): raise FileExistsError(f"Refusing to overwrite {ckpt}")
    out=RESULTS_ROOT/ACTIVE_STAGE; out.mkdir(parents=True,exist_ok=True)
    cmd=ds_prefix(1113)+["train_head.py","--model_name_or_path",DEBERTA_MODEL,"--instruction_train_data_path",str(ins),"--context_train_data_path",str(ctx),
        "--eval_data_path","data/crafted_instruction_data_squad_injection_qa.json","--bf16","--save_path",str(ckpt),"--max_epochs","1",
        "--train_batch_size","4","--micro_train_batch_size","4","--learning_rate","1e-5","--l2","0","--lr_scheduler","cosine",
        "--logging_steps","1","--adam_offload","--zero_stage","1","--seed",str(SEED),"--inject_rate","0.6","--log_file",str(out/"upstream.log")]
    last_return_code,last_stage_dir=run_stage(ACTIVE_STAGE,cmd,{"DS_SKIP_CUDA_CHECK":"1"})
    if last_return_code==0:
        weights=list(ckpt.glob("*.safetensors"))+list(ckpt.glob("*.bin"))
        if not (ckpt/"config.json").exists() or not weights: raise RuntimeError("Checkpoint validation failed")
        result={"status":"success","scientific_result":not smoke,"checkpoint":str(ckpt),"weight_files":[p.name for p in weights]}
        (last_stage_dir/"training_result.json").write_text(json.dumps(result,indent=2)+"\n",encoding="utf-8")
elif ACTIVE_STAGE.startswith("deberta_eval_") or ACTIVE_STAGE.startswith("generative_eval_"):
    family="deberta" if ACTIVE_STAGE.startswith("deberta") else "generative"; domain="squad" if ACTIVE_STAGE.endswith("squad") else "trivia"
    data,label=DATASETS[domain]; ckpt=DEBERTA_CHECKPOINT if family=="deberta" else GENERATIVE_CHECKPOINT; require_checkpoint(ckpt,family)
    out=RESULTS_ROOT/ACTIVE_STAGE; out.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,"run_detection.py","--model_path",str(ckpt),"--user_data_path",data,"--injected_instruction_data_path","data/crafted_instruction_data_alpaca.json",
         "--attack",*ATTACKS,"--sides",*POSITIONS,"--seed",str(SEED),"--log_path",str(out/"upstream.log")]
    last_return_code,last_stage_dir=run_stage(ACTIVE_STAGE,cmd)
    if last_return_code==0: parse_log("detection",last_stage_dir,label)
elif ACTIVE_STAGE in {"generative_train","extraction_train"}:
    gen=ACTIVE_STAGE=="generative_train"; script="train_classification.py" if gen else "train.py"; base=GENERATIVE_BASE_MODEL if gen else EXTRACTION_BASE_MODEL; ckpt=GENERATIVE_CHECKPOINT if gen else EXTRACTION_CHECKPOINT
    if ckpt.exists(): raise FileExistsError(f"Refusing to overwrite {ckpt}")
    out=RESULTS_ROOT/ACTIVE_STAGE; out.mkdir(parents=True,exist_ok=True)
    cmd=ds_prefix(1114 if gen else 1145)+[script,"--model_name_or_path",base,"--instruction_train_data_path","data/crafted_instruction_data_alpaca.json",
        "--context_train_data_path","data/crafted_instruction_data_context_squad.json","--eval_data_path","data/crafted_instruction_data_squad_injection_qa.json",
        "--bf16","--save_path",str(ckpt),"--max_epochs","1","--train_batch_size","4","--micro_train_batch_size","4","--learning_rate","1e-5",
        "--l2","0","--lr_scheduler","cosine","--logging_steps","1","--adam_offload","--zero_stage","1","--seed",str(SEED),"--log_file",str(out/"upstream.log")]
    if gen: cmd.extend(["--inject_rate","0.6"])
    last_return_code,last_stage_dir=run_stage(ACTIVE_STAGE,cmd,{"DS_SKIP_CUDA_CHECK":"1"})
elif ACTIVE_STAGE.startswith("removal_"):
    method="cls" if "segmentation" in ACTIVE_STAGE else "ext"; domain="squad" if ACTIVE_STAGE.endswith("squad") else "trivia"; data,label=DATASETS[domain]
    require_checkpoint(DEBERTA_CHECKPOINT,"DeBERTa")
    if method=="ext": require_checkpoint(EXTRACTION_CHECKPOINT,"extraction")
    out=RESULTS_ROOT/ACTIVE_STAGE; out.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,"run_purify.py","--model_path",str(DEBERTA_CHECKPOINT),"--user_data_path",data,"--injected_instruction_data_path","data/crafted_instruction_data_alpaca.json",
         "--attack",*INJECTED_ATTACKS,"--sides",*POSITIONS,"--purify_method",method,"--seed",str(SEED),"--log_path",str(out/"upstream.log")]
    if method=="ext": cmd[4:4]=["--ext_model_path",str(EXTRACTION_CHECKPOINT)]
    last_return_code,last_stage_dir=run_stage(ACTIVE_STAGE,cmd)
    if last_return_code==0: parse_log("removal",last_stage_dir,label)
elif ACTIVE_STAGE.startswith("final_asr_"):
    method="cls" if "segmentation" in ACTIVE_STAGE else "ext"; domain="squad" if ACTIVE_STAGE.endswith("squad") else "trivia"; data,label=DATASETS[domain]
    require_checkpoint(DEBERTA_CHECKPOINT,"DeBERTa"); require_checkpoint(EXTRACTION_CHECKPOINT,"extraction")
    out=RESULTS_ROOT/ACTIVE_STAGE; out.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,"run_evaluation_instruction.py","--model_path",TARGET_MODEL,"--filter_bot",str(DEBERTA_CHECKPOINT),"--extract_bot",str(EXTRACTION_CHECKPOINT),
         "--data_path",data,"--attack",*INJECTED_ATTACKS,"--defense","filter","--injection_type","adv","--purify_method",method,"--side",*POSITIONS,
         "--seed",str(SEED),"--log_path",str(out/"upstream.log")]
    last_return_code,last_stage_dir=run_stage(ACTIVE_STAGE,cmd)
    if last_return_code==0: parse_log("asr",last_stage_dir,label)
else: raise AssertionError(f"No dispatch branch for {ACTIVE_STAGE}")
print("Stage return code:",last_return_code)

Running deberta_smoke; full output -> /kaggle/working/base-paper-results/deberta_smoke/console.log
{
  "stage": "deberta_smoke",
  "status": "success",
  "return_code": 0,
  "finished_at_utc": "2026-10-07T16:47:24.237017+00:00"
}
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   elasticity_enabled ........... False
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   flops_profiler_config ........ {
    "enabled": false, 
    "recompute_fwd_factor": 0.0, 
    "profile_step": 1, 
    "module_depth": -1, 
    "top_modules": 1, 
    "detailed": true, 
    "output_file": null
}
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   fp16_auto_cast ............... None
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   fp16_enabled ................. False
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   fp16_master_weights_and_gradients  False
[2026-10-07 16:47:10,240] [INFO] [config.py:1003:print]   global_rank .................. 0
[2026-10-07 16:47:10,240] [

## Result viewer

In [7]:
if last_stage_dir is None: print("No result directory was created.")
else:
    print("Result directory:",last_stage_dir)
    for name in ["status.json","training_result.json","config.json","error.txt","detection_results.json","removal_results.json","asr_results.json","data_summary.json","environment.json","command.txt"]:
        p=last_stage_dir/name
        if p.exists(): print(f"\n===== {name} ====="); print(p.read_text(encoding="utf-8",errors="replace"))
    for name in ["console.log","upstream.log"]:
        p=last_stage_dir/name
        if p.exists(): print(f"\n===== final 120 lines: {name} ====="); print(tail_text(p,120))

Result directory: /kaggle/working/base-paper-results/deberta_smoke

===== status.json =====
{
  "stage": "deberta_smoke",
  "status": "success",
  "return_code": 0,
  "finished_at_utc": "2026-10-07T16:47:24.237017+00:00"
}


===== training_result.json =====
{
  "status": "success",
  "scientific_result": false,
  "checkpoint": "/kaggle/working/base-paper-models/deberta-smoke",
  "weight_files": [
    "model.safetensors"
  ]
}


===== config.json =====
{
  "stage": "deberta_smoke",
  "official_repository": "https://github.com/LukeChen-go/indirect-pia-detection.git",
  "official_commit": "1e9e0f00179adc154e9ca991b9cd3206c99f9588",
  "seed": 42,
  "started_at_utc": "2026-10-07T16:45:37.858470+00:00",
  "command": "/usr/bin/python3 -m deepspeed.launcher.runner --include localhost:0 --master_port 1113 train_head.py --model_name_or_path microsoft/deberta-v3-base --instruction_train_data_path /kaggle/working/base-paper-smoke-data/alpaca_64.json --context_train_data_path /kaggle/working/base-p

## Saving and continuing

Use Kaggle **Save Version** after the selected stage finishes. Keep notebook outputs. For training stages, also save `/kaggle/working/base-paper-models/` as Kaggle output so the checkpoint can be attached to a later session.

Recommended order: `audit` → `deberta_smoke` → `deberta_full` → DeBERTa evaluations → generative detector → extraction model → removal evaluations → final ASR. Change only `ACTIVE_STAGE` between runs and review each result before continuing.